In [1]:
import gzip
import random
import scipy
import numpy
from collections import defaultdict
from implicit import bpr
from surprise import SVD, Reader, Dataset
from surprise.model_selection import train_test_split
from tqdm import tqdm as tqdm

/workspaces/cse-258-assignment-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataDir = "./data"

In [3]:
def latentFactorModel(trainFile):
    tempPath = dataDir + "/temp.csv"
    with open(trainFile, "r") as a, open(tempPath, "w") as b:
        a.readline()
        for l in a.readlines():
            b.write(l)
        # Need to delete the header

    reader = Reader(line_format="user item timestamp rating", sep=",")
    data = Dataset.load_from_file(tempPath, reader=reader)

    model = SVD(n_factors=1, n_epochs=50)
    trainset, testset = train_test_split(data, test_size=0.25)
    model.fit(trainset)

    predictions = model.test(testset)

    sse = 0
    for p in predictions:
        sse += (p.r_ui - p.est) ** 2

    print("Error on own test set = ", str(sse / len(predictions)))

    return trainset, model

In [4]:
def testPredictionsLFM(trainset, model, testIn, testOut):
    with open(testIn, "r") as a, open(testOut, "w") as b:
        header = a.readline()
        b.write(header)
        for l in a.readlines():
            fields = l.split(',')
            u, i = fields[0].strip(), fields[1].strip()
            try:
                # print((u,i))
                est = model.predict(u, i).est
                b.write(l.strip() + ',' + str(est) + '\n')
            except Exception as e:
                print(e)

In [5]:
def BPRModel(trainFile):
    data = []
    userIDs, itemIDs = {}, {}
    with open(trainFile, "r") as a:
        a.readline()  # header
        for l in a:
            fields = l.split(",")
            u, i = fields[0].strip(), fields[1].strip()
            data.append((u, i))
            if u not in userIDs:
                userIDs[u] = len(userIDs)
            if i not in itemIDs:
                itemIDs[i] = len(itemIDs)

    Xui = scipy.sparse.lil_matrix((len(userIDs), len(itemIDs)))
    for u, i in data:
        Xui[userIDs[u], itemIDs[i]] = 1

    model = bpr.BayesianPersonalizedRanking(factors=5)
    model.fit(scipy.sparse.csr_matrix(Xui))

    return model, userIDs, itemIDs

In [6]:
def testPredictionsBPR(model, userIDs, itemIDs, testIn, testOut):

    with open(testIn, "r") as a, open(testOut, "w") as b:
        header = a.readline()
        b.write(header)
        randomGuesses = 0
        for l in a.readlines():
            fields = l.split(",")
            # User is first field, items are second last and last fields
            u, i1, i2 = fields[0].strip(), fields[-2].strip(), fields[-1].strip()
            try:
                u_ = userIDs[u]
                i1_ = itemIDs[i1]
                i2_ = itemIDs[i2]
            except Exception as e:
                # print("Missing user or item, guessing randomly")
                b.write(l.strip() + "," + random.choice([i1, i2]) + "\n")
                randomGuesses += 1
            try:
                uf = model.user_factors[u_]
                i1f = model.item_factors[i1_]
                i2f = model.item_factors[i2_]
                est = i1
                if uf.dot(i2f) > uf.dot(i1f):
                    est = i2
                b.write(l.strip() + "," + est + "\n")
            except Exception as e:
                print(e)

    print("Guessed randomly " + str(randomGuesses) + " times")

In [7]:
# Task 1: latent factor model

In [8]:
trainset, m1 = latentFactorModel(dataDir + "/" + "tasks_1_2_train.csv")

Error on own test set =  0.7860714161223766


In [9]:
testPredictionsLFM(
    trainset,
    m1,
    dataDir + "/" + "task_1_test.csv",
    dataDir + "/" + "task_1_predictions.csv",
)

In [10]:
# Task 2: BPR

In [11]:
m2, userIDs, itemIDs = BPRModel(dataDir + "/" + "tasks_1_2_train.csv")

  0%|                                                                                                                                                                                                                                                                                                                     | 0/100 [00:00<?, ?it/s]

  1%|██▋                                                                                                                                                                                                                                                                         | 1/100 [00:00<00:01, 55.82it/s, train_auc=50.25%, skipped=1.97%]

  2%|█████▎                                                                                                                                                                                                                                                                      | 2/100 [00:00<00:01, 53.76it/s, train_auc=50.40%, skipped=1.95%]

  3%|████████                                                                                                                                                                                                                                                                    | 3/100 [00:00<00:01, 55.48it/s, train_auc=50.37%, skipped=1.97%]

  4%|██████████▋                                                                                                                                                                                                                                                                 | 4/100 [00:00<00:01, 55.02it/s, train_auc=50.51%, skipped=1.97%]

  5%|█████████████▍                                                                                                                                                                                                                                                              | 5/100 [00:00<00:01, 49.84it/s, train_auc=50.51%, skipped=1.97%]

... (110 outputs left out)

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 99/100 [00:02<00:00, 38.76it/s, train_auc=89.87%, skipped=1.96%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 41.73it/s, train_auc=89.87%, skipped=1.96%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 41.73it/s, train_auc=89.83%, skipped=1.96%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 46.36it/s, train_auc=89.83%, skipped=1.96%]

In [12]:
testPredictionsBPR(
    m2,
    userIDs,
    itemIDs,
    dataDir + "/" + "task_2_test.csv",
    dataDir + "/" + "task_2_predictions.csv",
)

Guessed randomly 0 times


In [13]:
# Task 3: latent factor model, just ignore the timestamp

In [14]:
trainset, m3 = latentFactorModel(dataDir + "/" + "tasks_3_4_train.csv")

Error on own test set =  0.9388848118292432


In [15]:
testPredictionsLFM(
    trainset,
    m3,
    dataDir + "/" + "task_3_test.csv",
    dataDir + "/" + "task_3_predictions.csv",
)  # Don't use the timestamp

In [16]:
# Task 4: BPR, just ignore the timestamp

In [17]:
m4, userIDs, itemIDs = BPRModel(dataDir + "/" + "tasks_3_4_train.csv")

  0%|                                                                                                                                                                                                                                                                                                                     | 0/100 [00:00<?, ?it/s]

  1%|██▋                                                                                                                                                                                                                                                                         | 1/100 [00:00<00:02, 39.40it/s, train_auc=51.11%, skipped=5.13%]

  2%|█████▎                                                                                                                                                                                                                                                                      | 2/100 [00:00<00:03, 32.56it/s, train_auc=51.20%, skipped=5.14%]

  3%|████████                                                                                                                                                                                                                                                                    | 3/100 [00:00<00:02, 39.51it/s, train_auc=51.34%, skipped=5.13%]

  4%|██████████▋                                                                                                                                                                                                                                                                 | 4/100 [00:00<00:02, 45.05it/s, train_auc=51.31%, skipped=5.14%]

  5%|█████████████▍                                                                                                                                                                                                                                                              | 5/100 [00:00<00:02, 45.73it/s, train_auc=51.31%, skipped=5.14%]

... (108 outputs left out)

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 99/100 [00:01<00:00, 49.80it/s, train_auc=84.31%, skipped=5.11%]

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 99/100 [00:01<00:00, 49.80it/s, train_auc=84.28%, skipped=5.14%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:01<00:00, 49.80it/s, train_auc=84.30%, skipped=5.13%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:01<00:00, 53.14it/s, train_auc=84.30%, skipped=5.13%]

In [18]:
testPredictionsBPR(
    m4,
    userIDs,
    itemIDs,
    dataDir + "/" + "task_4_test.csv",
    dataDir + "/" + "task_4_predictions.csv",
)

Guessed randomly 0 times


In [19]:
# Task 5: BPR, ignore timestamp/ordering

In [20]:
m5, userIDs, itemIDs = BPRModel(dataDir + "/" + "task_5_train.csv")

  0%|                                                                                                                                                                                                                                                                                                                     | 0/100 [00:00<?, ?it/s]

  1%|██▋                                                                                                                                                                                                                                                                         | 1/100 [00:00<00:02, 35.09it/s, train_auc=50.25%, skipped=1.84%]

  2%|█████▎                                                                                                                                                                                                                                                                      | 2/100 [00:00<00:02, 39.57it/s, train_auc=50.29%, skipped=1.83%]

  3%|████████                                                                                                                                                                                                                                                                    | 3/100 [00:00<00:02, 39.60it/s, train_auc=50.39%, skipped=1.84%]

  4%|██████████▋                                                                                                                                                                                                                                                                 | 4/100 [00:00<00:02, 38.62it/s, train_auc=50.39%, skipped=1.84%]

  4%|██████████▋                                                                                                                                                                                                                                                                 | 4/100 [00:00<00:02, 38.62it/s, train_auc=50.52%, skipped=1.84%]

... (117 outputs left out)

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 99/100 [00:03<00:00, 27.54it/s, train_auc=89.47%, skipped=1.84%]

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 99/100 [00:03<00:00, 27.54it/s, train_auc=89.50%, skipped=1.84%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 27.54it/s, train_auc=89.52%, skipped=1.83%]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 31.10it/s, train_auc=89.52%, skipped=1.83%]

In [21]:
testPredictionsBPR(
    m5,
    userIDs,
    itemIDs,
    dataDir + "/" + "task_5_test.csv",
    dataDir + "/" + "task_5_predictions.csv",
)

Guessed randomly 3441 times


In [22]:
# Task 6: compute median delta

In [23]:
with open(dataDir + "/" + "task_6_train.csv") as a:
    a.readline()
    userDeltas = defaultdict(list)
    uLast = {}  # Time of user's most recent interaction
    for l in a.readlines():
        u, i, t, r = l.strip().split(",")
        t = float(t)
        if u in uLast:
            userDeltas[u].append(t - uLast[u])
        uLast[u] = t

medians = {}
for u in userDeltas:
    medians[u] = numpy.median(userDeltas[u])

with (
    open(dataDir + "/" + "task_6_test.csv", "r") as a,
    open(dataDir + "/" + "task_6_predictions.csv", "w") as b,
):
    b.write(a.readline())  # header
    for l in a.readlines():
        u, i = l.strip().split(",")
        b.write(u + "," + i + "," + str(medians[u]) + "\n")

In [24]:
# Baseline performance:

# public =  {"task1": 0.7429, "task2":0.8716, "task3":0.9820, "task4":0.8036, "task5":0.8189, "task6":13563799}
# private = {"task1": 0.7738, "task2":0.8736, "task3":0.9582, "task4":0.8007, "task5":0.8195, "task6":13481806}